In [5]:
# ============================================================
# PHASE 2 - FULL ADVERSARIAL ATTACK EVALUATION
# VS Code Jupyter Version
#
# SETUP BEFORE RUNNING:
#   1. Install dependencies:
#      pip install adversarial-robustness-toolbox[tensorflow]
#   2. Make sure these files are in the same folder as this notebook:
#        baseline_model.keras
#        scaler.pkl
#        imputer.pkl
#        label_encoder.pkl
#        X_test.npy
#        y_test.npy
#   3. Run cells in order
# ============================================================


# ── CELL 1: Check environment ────────────────────────────────
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)
if not gpus:
    print("\nNo GPU detected — running on CPU (slower but works fine)")
else:
    print("GPU confirmed — good to go!")


# ── CELL 2: Imports ──────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
import logging
import os
from sklearn.metrics import accuracy_score, classification_report

logging.getLogger('art').setLevel(logging.ERROR)

# ── CELL 3: Load Phase 1 artifacts ───────────────────────────
print("Loading Phase 1 artifacts...")

# Load full saved model directly
model = tf.keras.models.load_model('baseline_model.keras')
model.summary()

scaler  = joblib.load('scaler.pkl')
imputer = joblib.load('imputer.pkl')
le      = joblib.load('label_encoder.pkl')
X_test  = np.load('X_test.npy')
y_test  = np.load('y_test.npy')

print(f"\nTest set shape: {X_test.shape}")
print(f"Classes: {list(le.classes_)}")


# ── CELL 4: Build balanced evaluation subset ─────────────────
print("\nBuilding representative subset...")

SAMPLES_PER_CLASS = 2000
np.random.seed(42)
subset_indices = []

for class_idx in np.unique(y_test):
    class_indices = np.where(y_test == class_idx)[0]
    n_samples = min(SAMPLES_PER_CLASS, len(class_indices))
    chosen = np.random.choice(class_indices, size=n_samples, replace=False)
    subset_indices.extend(chosen)

np.random.shuffle(subset_indices)
X_subset = X_test[subset_indices]
y_subset = y_test[subset_indices]

print(f"Subset: {X_subset.shape[0]} samples ({SAMPLES_PER_CLASS} per class)")

# Clean predictions — needed for attack success rate calculation
y_pred_clean   = model.predict(X_subset, verbose=0).argmax(axis=1)
clean_correct  = y_pred_clean == y_subset
baseline_acc   = accuracy_score(y_subset, y_pred_clean)

print(f"\nBaseline accuracy on subset: {baseline_acc:.4f}")
print("\nBaseline per-class performance:")
print(classification_report(y_subset, y_pred_clean,
                             target_names=le.classes_, zero_division=0))


# ── CELL 5: Wrap model for ART ───────────────────────────────
from art.estimators.classification import TensorFlowV2Classifier

loss_fn    = tf.keras.losses.SparseCategoricalCrossentropy()
classifier = TensorFlowV2Classifier(
    model=model,
    nb_classes=len(le.classes_),
    input_shape=(X_subset.shape[1],),
    loss_object=loss_fn,
    clip_values=(float(X_subset.min()), float(X_subset.max()))
)
print("Model wrapped for ART successfully")


# ── Helper: evaluate attack results ──────────────────────────
def evaluate_attack(y_true, y_pred_adv, clean_correct, attack_name, eps=None):
    acc = accuracy_score(y_true, y_pred_adv)
    n_clean_correct = np.sum(clean_correct)
    attack_success  = (
        np.sum(clean_correct & (y_pred_adv != y_true)) / n_clean_correct
        if n_clean_correct > 0 else 0.0
    )
    eps_label = f"ε={eps}" if eps is not None else "N/A"
    print(f"\n  [{attack_name} {eps_label}]")
    print(f"  Accuracy:        {acc:.4f}  (baseline: {baseline_acc:.4f})")
    print(f"  Attack success:  {attack_success:.4f}")
    print(f"  Per-class recall:")
    for class_idx in np.unique(y_true):
        mask      = y_true == class_idx
        class_acc = accuracy_score(y_true[mask], y_pred_adv[mask])
        print(f"    {le.classes_[class_idx]:<20} {class_acc:.4f}")
    return acc, attack_success


# Results storage
results  = []
epsilons = [0.01, 0.05, 0.1, 0.15, 0.2]


# ── CELL 6: FGSM Attack ──────────────────────────────────────
from art.attacks.evasion import FastGradientMethod

print("\n" + "="*50)
print("RUNNING FGSM ATTACK")
print("="*50)

for eps in epsilons:
    print(f"\nFGSM ε={eps}...")
    fgsm  = FastGradientMethod(estimator=classifier, eps=eps)
    X_adv = fgsm.generate(x=X_subset, batch_size=512)
    y_pred_adv = model.predict(X_adv, verbose=0).argmax(axis=1)
    acc, attack_success = evaluate_attack(
        y_subset, y_pred_adv, clean_correct, "FGSM", eps
    )
    results.append({
        'Attack': 'FGSM', 'Epsilon': eps,
        'Accuracy': acc, 'Attack_Success_Rate': attack_success,
        'Avg_Perturbation': 'N/A'
    })

print("\nFGSM complete.")


# ── CELL 7: BIM Attack ───────────────────────────────────────
from art.attacks.evasion import BasicIterativeMethod

print("\n" + "="*50)
print("RUNNING BIM ATTACK")
print("="*50)

for eps in epsilons:
    print(f"\nBIM ε={eps}...")
    bim = BasicIterativeMethod(
        estimator=classifier,
        eps=eps,
        eps_step=eps / 10,
        max_iter=10
    )
    X_adv = bim.generate(x=X_subset, batch_size=512)
    y_pred_adv = model.predict(X_adv, verbose=0).argmax(axis=1)
    acc, attack_success = evaluate_attack(
        y_subset, y_pred_adv, clean_correct, "BIM", eps
    )
    results.append({
        'Attack': 'BIM', 'Epsilon': eps,
        'Accuracy': acc, 'Attack_Success_Rate': attack_success,
        'Avg_Perturbation': 'N/A'
    })

print("\nBIM complete.")


# ── CELL 8: PGD Attack ───────────────────────────────────────
from art.attacks.evasion import ProjectedGradientDescent

print("\n" + "="*50)
print("RUNNING PGD ATTACK")
print("="*50)

for eps in epsilons:
    print(f"\nPGD ε={eps}...")
    pgd = ProjectedGradientDescent(
        estimator=classifier,
        eps=eps,
        eps_step=eps / 10,
        max_iter=10,
        num_random_init=5
    )
    X_adv = pgd.generate(x=X_subset, batch_size=256)
    y_pred_adv = model.predict(X_adv, verbose=0).argmax(axis=1)
    acc, attack_success = evaluate_attack(
        y_subset, y_pred_adv, clean_correct, "PGD", eps
    )
    results.append({
        'Attack': 'PGD', 'Epsilon': eps,
        'Accuracy': acc, 'Attack_Success_Rate': attack_success,
        'Avg_Perturbation': 'N/A'
    })

print("\nPGD complete.")


# ── CELL 9: C&W Attack ───────────────────────────────────────
from art.attacks.evasion import CarliniL2Method

print("\n" + "="*50)
print("RUNNING C&W ATTACK")
print("="*50)

# C&W is slow — use 100 samples per class (800 total)
CW_SAMPLES_PER_CLASS = 100
cw_indices = []

for class_idx in np.unique(y_subset):
    class_mask = np.where(y_subset == class_idx)[0]
    n          = min(CW_SAMPLES_PER_CLASS, len(class_mask))
    chosen     = np.random.choice(class_mask, size=n, replace=False)
    cw_indices.extend(chosen)

np.random.shuffle(cw_indices)
X_cw_subset = X_subset[cw_indices]
y_cw_subset = y_subset[cw_indices]

y_pred_cw_clean  = model.predict(X_cw_subset, verbose=0).argmax(axis=1)
clean_correct_cw = y_pred_cw_clean == y_cw_subset

print(f"C&W subset: {len(X_cw_subset)} samples ({CW_SAMPLES_PER_CLASS} per class)")

cw = CarliniL2Method(
    classifier=classifier,
    confidence=0.0,
    max_iter=50,
    learning_rate=0.01,
    binary_search_steps=3,
    batch_size=32            # default is 1 — 32 is 30x faster on GPU
)

print("Generating C&W adversarial examples (this may take a few minutes)...")
X_adv_cw   = cw.generate(x=X_cw_subset)
y_pred_adv = model.predict(X_adv_cw, verbose=0).argmax(axis=1)

acc, attack_success = evaluate_attack(
    y_cw_subset, y_pred_adv, clean_correct_cw, "C&W"
)

perturbation     = X_adv_cw - X_cw_subset
avg_perturbation = np.mean(np.linalg.norm(perturbation, axis=1))
print(f"\n  Average perturbation magnitude: {avg_perturbation:.6f}")
print(f"  (Smaller = more subtle/realistic attack)")

results.append({
    'Attack': 'C&W', 'Epsilon': 'N/A',
    'Accuracy': acc, 'Attack_Success_Rate': attack_success,
    'Avg_Perturbation': avg_perturbation
})

print("\nC&W complete.")


# ── CELL 10: Save all results ─────────────────────────────────
print("\n" + "="*50)
print("SAVING ALL RESULTS")
print("="*50)

results_df = pd.DataFrame(results)
print("\nFull results summary:")
print(results_df.to_string(index=False))

results_df.to_csv('phase2_results.csv', index=False)
np.save('X_adv_cw.npy',    X_adv_cw)
np.save('X_cw_subset.npy', X_cw_subset)
np.save('y_cw_subset.npy', y_cw_subset)

print("\nFiles saved:")
print("  phase2_results.csv   — full results table")
print("  X_adv_cw.npy         — C&W adversarial examples (needed for Phase 3)")
print("  X_cw_subset.npy      — clean subset used for C&W")
print("  y_cw_subset.npy      — labels for C&W subset")
print("\nPhase 2 complete!")

TensorFlow version: 2.16.2
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU confirmed — good to go!
Loading Phase 1 artifacts...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        10,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │           264 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 164,794 (643.73 KB)

 Trainable params: 54,632 (213.41 KB)

...

PGD - Batches: 0it [00:00, ?it/s]

2026-03-20 11:18:36.221072: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



  [PGD ε=0.2]
  Accuracy:        0.5549  (baseline: 0.5923)
  Attack success:  0.2146
  Per-class recall:
    Benign               0.7640
    Brute_Force          0.3087
    DDoS                 0.6975
    DoS                  0.5210
    Mirai                0.9880
    Reconnaissance       0.4955
    Spoofing             0.3810
    Web_Attack           0.1351

PGD complete.

RUNNING C&W ATTACK
C&W subset: 800 samples (100 per class)
Generating C&W adversarial examples (this may take a few minutes)...


C&W L_2:   0%|          | 0/25 [00:00<?, ?it/s]


  [C&W N/A]
  Accuracy:        0.5487  (baseline: 0.5923)
  Attack success:  0.0023
  Per-class recall:
    Benign               0.6100
    Brute_Force          0.2900
    DDoS                 0.7100
    DoS                  0.7400
    Mirai                0.9900
    Reconnaissance       0.5800
    Spoofing             0.4000
    Web_Attack           0.0700

  Average perturbation magnitude: 0.000016
  (Smaller = more subtle/realistic attack)

C&W complete.

SAVING ALL RESULTS

Full results summary:
Attack Epsilon  Accuracy  Attack_Success_Rate Avg_Perturbation
  FGSM    0.01  0.591628             0.007464              N/A
  FGSM    0.05  0.583255             0.052132              N/A
  FGSM     0.1  0.572605             0.099966              N/A
  FGSM    0.15  0.560683             0.151985              N/A
  FGSM     0.2  0.554119             0.192582              N/A
   BIM    0.01  0.591494             0.007577              N/A
   BIM    0.05  0.581782             0.054619        